In [1]:
3+5

8

In [5]:
import requests
import pandas as pd
import time
from tqdm import tqdm

API_KEY = "7b60f09e5f7e4c6694ffdd4cb5437c9f"

START = (pd.Timestamp.utcnow() - pd.Timedelta(days=180)).strftime("%Y-%m-%d")
END   = pd.Timestamp.utcnow().strftime("%Y-%m-%d")

SYMBOLS = {
    # crypto
    "BTC": "BTC/USD",
    "ETH": "ETH/USD",
    "SOL": "SOL/USD",

    # macro proxies
    "SPY": "SPY",
    "DXY": "UUP",
    "GOLD": "XAU/USD",
}


def download_td_15m(symbol, start, end):
    url = "https://api.twelvedata.com/time_series"

    start_dt = pd.Timestamp(start)
    end_dt   = pd.Timestamp(end)

    chunk = pd.Timedelta(days=50)  # <5000 свечей
    parts = []

    while start_dt < end_dt:
        chunk_end = min(start_dt + chunk, end_dt)

        params = {
            "symbol": symbol,
            "interval": "15min",
            "start_date": start_dt.strftime("%Y-%m-%d"),
            "end_date": chunk_end.strftime("%Y-%m-%d"),
            "apikey": API_KEY,
            "outputsize": 5000,
            "format": "JSON",
        }

        r = requests.get(url, params=params, timeout=30)
        js = r.json()

        if "values" in js:
            df = pd.DataFrame(js["values"])
            df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
            df = df.set_index("datetime").sort_index()
            df = df.astype(float)
            df = df[["open", "high", "low", "close"]]
            df = df.add_prefix(f"{name}_")
            parts.append(df)
        else:
            print("❌", symbol, js.get("message"))

        start_dt = chunk_end
        time.sleep(8)  # важно для free

    if not parts:
        return pd.Series(dtype=float)

    s = pd.concat(parts)
    s = s[~s.index.duplicated()].sort_index()
    return s


# ---------- MAIN ----------
all_series = []

for name, symbol in SYMBOLS.items():
    print("Downloading", name)
    s = download_td_15m(symbol, START, END)
    s.name = name
    all_series.append(s)

macro_df = pd.concat(all_series, axis=1)


# 🔧 убираем дыры
macro_df = macro_df.ffill().dropna()

macro_df.to_pickle("macro_crypto_15m.pickle")

print("✅ Saved:", macro_df.shape)

✅ Saved: (17243, 24)


In [2]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from tqdm import tqdm

BASE_URL = "https://api.binance.com/api/v3/klines"

# ====== НАСТРОЙКИ ======
SYMBOLS = [
    "BTCUSDT",
    "ETHUSDT",
    "SOLUSDT",
    "BNBUSDT",
    "XRPUSDT",
    "ADAUSDT",
    "DOGEUSDT",
    "AVAXUSDT",
]

INTERVAL = "15m"

END_DATE = datetime.utcnow()
START_DATE = END_DATE - timedelta(days=180)  # полгода


# ====== ФУНКЦИЯ СКАЧКИ ======
def download_klines(symbol, start, end):
    start_ts = int(pd.Timestamp(start, tz="UTC").timestamp() * 1000)
    end_ts   = int(pd.Timestamp(end, tz="UTC").timestamp() * 1000)

    all_rows = []
    current = start_ts

    print(f"🚀 Downloading {symbol}")

    while current < end_ts:
        params = {
            "symbol": symbol,
            "interval": INTERVAL,
            "startTime": current,
            "endTime": end_ts,
            "limit": 1000,
        }

        r = requests.get(BASE_URL, params=params)
        data = r.json()

        if not data:
            break

        all_rows.extend(data)
        current = data[-1][0] + 1

        time.sleep(0.2)  # анти-бан

    if not all_rows:
        return pd.DataFrame()

    cols = [
        "open_time","open","high","low","close","volume",
        "close_time","quote_asset_volume","number_of_trades",
        "taker_buy_base","taker_buy_quote","ignore"
    ]

    df = pd.DataFrame(all_rows, columns=cols)

    df["datetime"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df = df.set_index("datetime")

    float_cols = [
        "open","high","low","close","volume",
        "quote_asset_volume","taker_buy_base","taker_buy_quote"
    ]

    df[float_cols] = df[float_cols].astype(float)
    df["number_of_trades"] = df["number_of_trades"].astype(int)

    return df


# ====== MAIN ======
all_coins = []

for symbol in SYMBOLS:
    df = download_klines(symbol, START_DATE, END_DATE)

    if df.empty:
        continue

    # добавляем префикс монеты
    df = df.add_prefix(symbol + "_")
    all_coins.append(df)

    time.sleep(1)

# ====== MERGE ======
full = pd.concat(all_coins, axis=1).sort_index()

# общий календарь 15m
full = full.resample("15min").last()

full.to_pickle("crypto_multi_15m_6m.pickle")

print("✅ FINAL SHAPE:", full.shape)

🚀 Downloading BTCUSDT
🚀 Downloading ETHUSDT
🚀 Downloading SOLUSDT
🚀 Downloading BNBUSDT
🚀 Downloading XRPUSDT
🚀 Downloading ADAUSDT
🚀 Downloading DOGEUSDT
🚀 Downloading AVAXUSDT
✅ FINAL SHAPE: (17280, 96)
